In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data Preprocessing and Mapping

In [ ]:
import pandas as pd
from datasets import Dataset
from PIL import Image
import os

# --- 1. SET YOUR VARIABLES (You must change these!) ---

# This is the path to your 'Training' folder
BASE_TRAINING_FOLDER = '/content/drive/MyDrive/Drishti-GS1_files/Drishti-GS1_files/Training'

# --- 2. Define the exact paths from your structure ---

# FIX 1: Your diagram showed 'Images folder' (with a space)
IMAGE_FOLDER = os.path.join(BASE_TRAINING_FOLDER, 'Images')

# FIX 2: Your diagram showed 'GT Folder' (with a space)
GT_BASE_FOLDER = os.path.join(BASE_TRAINING_FOLDER, 'GT')


# --- 3. Loop through data and build a list ---

print("Starting to scan your dataset...")
data_list = []

# Get a list of all image filenames from your Images folder
try:
    image_filenames = os.listdir(IMAGE_FOLDER)
    print(f"Found {len(image_filenames)} images in {IMAGE_FOLDER}")
except FileNotFoundError:
    print(f"ERROR: Could not find the Image Folder at {IMAGE_FOLDER}")
    # Stop the script if the folder isn't found
    raise

# Loop through every image you found
for image_filename in image_filenames:
    if not image_filename.endswith('.png'):
        continue  # Skip any non-png files

    # --- A. Get the image path ---
    image_path = os.path.join(IMAGE_FOLDER, image_filename)

    # --- B. Find the matching ground truth ---

    # Get the base name, e.g., "dristiGS_002" from "dristiGS_002.png"
    base_name = image_filename.split('.')[0]

    # Construct the path to the .txt file, based on your structure
    # e.g., Training/GT Folder/dristiGS_002/dristiGS_002_cdrValues.txt
    cdr_file_path = os.path.join(GT_BASE_FOLDER, base_name, f"{base_name}_cdrValues.txt")

    # --- C. Read the CDR value from the .txt file ---
    try:
        with open(cdr_file_path, 'r') as f:
            cdr_value = f.read().strip() # Read the text, remove whitespace

        # Add the data to our list
        data_list.append({
            "image_path": image_path,
            "cdr_value": cdr_value
        })

    except FileNotFoundError:
        print(f"Warning: Could not find {cdr_file_path}. Skipping image {image_filename}.")
    except Exception as e:
        print(f"Error reading {cdr_file_path}: {e}")

print(f"\nSuccessfully found and paired {len(data_list)} images with their CDR values.")

# --- 4. Create the Hugging Face Dataset ---

# Create a Dataset object from our list
hf_dataset = Dataset.from_list(data_list)

# --- 5. Define the main processing function ---

def create_chat_format(row):
    """
    This function takes one row and formats it,
    but ONLY stores the IMAGE PATH (string), not the loaded image.
    """
    image_path = row['image_path'] # The path from our data list
    cdr_answer = str(row['cdr_value']) # The answer

    # This is the VQA prompt
    prompt = "What is the cup-to-disc ratio for this fundus image?"

    # This is the "chat" format
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},

                # --- THIS IS THE FIX (You did this correctly!) ---
                # We are now storing the 'image_path' string
                {"type": "image", "image": image_path}
                # --- END OF FIX ---
            ]
        },
        {
            "role": "model",
            "content": [
                {"type": "text", "text": cdr_answer}
            ]
        }
    ]

    # We no longer need the try/except for Image.open here
    return {"messages": messages}

# --- FIX 3: REMOVED the stray 'except FileNotFoundError:' block ---
# The 'except' block was here, causing a SyntaxError. It's now gone.


# --- 6. Apply the formatting ---

print("Starting dataset processing... This might take a few minutes.")
# We pass image_path to keep it for loading, but remove it later
processed_dataset = hf_dataset.map(
    create_chat_format,
    remove_columns=['image_path', 'cdr_value'] # Clean up old columns
)

# Remove any rows that failed (though none should fail now)
processed_dataset = processed_dataset.filter(lambda example: example["messages"] is not None)

print("Processing complete!")

# --- 7. Check your work ---
print("\nHere is an example of your first processed data entry:")
print(processed_dataset[0]['messages'])

# This 'processed_dataset' object is what you will feed to the SFTTrainer.
# To save to disk (optional):
# processed_dataset.save_to_disk('./my_processed_glaucoma_vqa_dataset')

Starting to scan your dataset...
Found 50 images in /content/drive/MyDrive/Drishti-GS1_files/Drishti-GS1_files/Training/Images

Successfully found and paired 50 images with their CDR values.
Starting dataset processing... This might take a few minutes.


Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Filter:   0%|          | 0/50 [00:00<?, ? examples/s]

Processing complete!

Here is an example of your first processed data entry:
[{'content': [{'image': None, 'text': 'What is the cup-to-disc ratio for this fundus image?', 'type': 'text'}, {'image': '/content/drive/MyDrive/Drishti-GS1_files/Drishti-GS1_files/Training/Images/drishtiGS_002.png', 'text': None, 'type': 'image'}], 'role': 'user'}, {'content': [{'image': None, 'text': '0.76 0.77 0.79 0.75', 'type': 'text'}], 'role': 'model'}]


In [ ]:
!pip install -U bitsandbytes
!pip install transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.2 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

print("--- T4 GPU Detected: Forcing 4-bit load with float16 compute ---")

# 1. Define 4-bit configuration (Crucial for memory)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    # We MUST use float16 here. float32 is too big for T4 memory.
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load Model
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    torch_dtype=torch.float16, # Match the compute dtype
    device_map="auto",
    # use_cpu_initialization=True  <-- REMOVED (was causing error)
)

# 3. Load Processor
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

print("✅ Model loaded on T4.")

--- T4 GPU Detected: Forcing 4-bit load with float16 compute ---


config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

✅ Model loaded on T4.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. Prepare model for training (saves memory)
model = prepare_model_for_kbit_training(model)

# 2. Define LoRA Config
peft_config = LoraConfig(
    r=8,          # Reduced from 16 to 8 to save a tiny bit more memory
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM"
)

# 3. Apply LoRA to model
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 19,248,896 || all params: 4,319,328,368 || trainable%: 0.4456


In [ ]:
from typing import Any
from PIL import Image

def collate_fn(examples: list[dict[str, Any]]):
    texts = []
    images = []

    for example in examples:
        messages = example["messages"]

        # 1. Process text
        texts.append(processor.apply_chat_template(
            messages, add_generation_prompt=False, tokenize=False
        ).strip())

        # 2. Process image (load from path)
        image_content = None
        for content_part in messages[0]['content']:
            if content_part['type'] == 'image':
                image_content = content_part['image']
                break

        if image_content:
            images.append([Image.open(image_content).convert("RGB")])
        else:
            images.append([Image.new('RGB', (100, 100))]) # Fallback

    # 3. Tokenize and batch
    batch = processor(text=texts, images=images, return_tensors="pt", padding=True)

    # 4. Create labels (masking special tokens)
    labels = batch["input_ids"].clone()
    image_token_id = processor.tokenizer.convert_tokens_to_ids(
        processor.tokenizer.special_tokens_map["boi_token"]
    )
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels
    return batch

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.8/462.8 kB 13.6 MB/s eta 0:00:00


In [ ]:
from trl import SFTConfig, SFTTrainer

# --- MASTER SETTINGS FOR T4 STABILITY ---
args = SFTConfig(
    output_dir="medgemma-t4-stable",
    num_train_epochs=20,
    learning_rate=2e-4,

    # Memory Settings (AGGRESSIVE)
    per_device_train_batch_size=1,      # Must be 1 on T4
    gradient_accumulation_steps=16,     # Accumulate to simulate batch size 16
    gradient_checkpointing=True,        # Mandatory for T4
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Fixes some instability

    # Stability Settings
    bf16=False,                         # T4 cannot do bf16
    fp16=True,                          # MUST be True for memory. We fight instability with other settings.
    optim="adamw_torch",                # Standard optimizer, often more stable than 8-bit versions
    max_grad_norm=0.3,                  # Clips huge gradients that cause "unscale" errors

    # Data Settings
    max_length=512,                     # Keep this small (512 is safe, 256 if it still crashes)
    dataset_text_field="messages",
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    packing=False,                      # Ensure packing is OFF to save memory

    # Logging
    logging_steps=50,
    save_strategy="no",                 # Don't save during training to save disk IO/time
    eval_strategy="no",
    report_to="none",                   # Turn off simple reporting for now
    push_to_hub=False
)

# --- Trainer Init (Using latest library terms) ---
trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=processed_dataset,
    data_collator=collate_fn,
    processing_class=processor,         # The new, correct term for 'tokenizer'
    peft_config=peft_config
)

# --- START TRAINING ---
print("Starting stable T4 training...")
trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Starting stable T4 training...


Step,Training Loss


Step,Training Loss
50,1.401900


TrainOutput(global_step=80, training_loss=0.9103003084659577, metrics={'train_runtime': 4357.853, 'train_samples_per_second': 0.229, 'train_steps_per_second': 0.018, 'total_flos': 6667343851680000.0, 'train_loss': 0.9103003084659577, 'entropy': 0.6822832701322825, 'num_tokens': 305000.0, 'mean_token_accuracy': 0.9584875165120416, 'epoch': 20.0})

# **Save Model **

In [ ]:
# 1. Define where to save (e.g., in your Drive so it's permanent)
# Make sure this folder exists or Colab can create it
save_path = "/content/drive/MyDrive/medgemma_glaucoma_v2_epoch20"

# 2. Save the adapter model
trainer.save_model(save_path)
processor.save_pretrained(save_path)

print(f"✅ Model successfully saved to: {save_path}")

✅ Model successfully saved to: /content/drive/MyDrive/medgemma_glaucoma_v2_epoch20


Final Step: Test It (Inference)
Now, let's see if it actually learned anything. We will load the base model again, merge your new saved adapter onto it, and ask it a question about one of your images.

Open a new cell and run this complete inference script. Change TEST_IMAGE_PATH to an actual image from your Images folder.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from PIL import Image

# --- 1. SETUP ---
BASE_MODEL_ID = "google/medgemma-4b-it"
ADAPTER_PATH = "/content/drive/MyDrive/medgemma_glaucoma_v2_epoch20"
# UPDATE THIS TO YOUR TEST IMAGE PATH
TEST_IMAGE_PATH = "/content/drive/MyDrive/Drishti-GS1_files/Drishti-GS1_files/Test/Images/drishtiGS_003.png"

# --- 2. LOAD BASE MODEL (Forced to GPU 0 for stability) ---
print("Loading base model directly to GPU 0...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0} # Force to GPU 0, avoids "meta tensor" error
)
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)

# --- 3. MERGE ADAPTER ---
print("Loading your FINAL 20-epoch adapter...")
model.load_adapter(ADAPTER_PATH)
print("✅ Adapter merged!")

# --- 4. INFERENCE ---
image = Image.open(TEST_IMAGE_PATH).convert("RGB")
prompt = "What is the cup-to-disc ratio for this fundus image?"
messages = [{"role": "user", "content": [{"type": "text", "text": prompt}, {"type": "image", "image": image}]}]

print("Generating...")
inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt").to(model.device)

# --- THE FIX ---
# We define a list of "banned" token lists.
# This bans BOTH the <pad> token AND the <eos> token.
# The model is now FORCED to output something else.
banned_tokens = [
    [processor.tokenizer.pad_token_id],
    [processor.tokenizer.eos_token_id]
]
# --- END OF FIX ---

outputs = model.generate(
    **inputs,
    max_new_tokens=20,
    do_sample=False,
    bad_words_ids=banned_tokens  # Apply our new ban list
)

print("\n--- FINAL MODEL PREDICTION ---")
# Use the "raw" decode to see exactly what it says
print(processor.decode(outputs[0], skip_special_tokens=True).split("model\n")[-1].strip())

Loading base model directly to GPU 0...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading your FINAL 20-epoch adapter...
✅ Adapter merged!
Generating...

--- FINAL MODEL PREDICTION ---

